In [1]:
#data structures
import pandas as pd
import numpy as np

#machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler#needed with PCA
from sklearn.preprocessing import TargetEncoder

from sklearn.pipeline import Pipeline

import prince #used for MCA (multiple correspondence analysis)

#metrics (performace and machine learning scores)
from sklearn.metrics import roc_auc_score
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import time #used for seeing how long it takes to run programs

np.random.seed(42)

In [2]:
import os
print(os.getcwd())

D:\Code\Earthquake


In [3]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

In [5]:
#experimental
# columnList = df.drop(['damage_grade', 'building_id'], axis = 1).columns.tolist()
# df = df.drop_duplicates(subset=columnList, keep=False, inplace=False)
# df.info()




columnList = df.drop(['damage_grade', 'building_id'], axis = 1).columns.tolist()

duplicates = df[df.duplicated(subset=columnList, keep=False)]

mostCommon = (
    duplicates.groupby(columnList)['damage_grade']
      .agg(lambda x: x.value_counts().idxmax())
      .reset_index()
)

mostCommon.sort_values(by=columnList).head(10)

#remove duplicates
df = df.drop_duplicates(subset=columnList, keep=False, inplace=False)
# df.info()


#add back the most common without adding back the outliers
df = pd.concat([df, mostCommon]) 
# df2 = df.copy()
# print(duplicates.shape)
# print(mostCommon.shape)
# print(df2.shape)
# df2 = df2.fillna(0)
# print(df.shape)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 244410 entries, 0 to 12352
Data columns (total 40 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   building_id                             232057 non-null  float64
 1   geo_level_1_id                          244410 non-null  int64  
 2   geo_level_2_id                          244410 non-null  int64  
 3   geo_level_3_id                          244410 non-null  int64  
 4   count_floors_pre_eq                     244410 non-null  int64  
 5   age                                     244410 non-null  int64  
 6   area_percentage                         244410 non-null  int64  
 7   height_percentage                       244410 non-null  int64  
 8   land_surface_condition                  244410 non-null  object 
 9   foundation_type                         244410 non-null  object 
 10  roof_type                               244410 non

In [6]:
print(df.columns.tolist())

['building_id', 'geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id', 'count_floors_pre_eq', 'age', 'area_percentage', 'height_percentage', 'land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'has_superstructure_adobe_mud', 'has_superstructure_mud_mortar_stone', 'has_superstructure_stone_flag', 'has_superstructure_cement_mortar_stone', 'has_superstructure_mud_mortar_brick', 'has_superstructure_cement_mortar_brick', 'has_superstructure_timber', 'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered', 'has_superstructure_rc_engineered', 'has_superstructure_other', 'legal_ownership_status', 'count_families', 'has_secondary_use', 'has_secondary_use_agriculture', 'has_secondary_use_hotel', 'has_secondary_use_rental', 'has_secondary_use_institution', 'has_secondary_use_school', 'has_secondary_use_industry', 'has_secondary_use_health_post', 'has_secondary_use_gov_office', 'has_secondary_use_use_police

In [10]:
#Sampling the data so test runs don't take so long
dataSample = df.sample(frac=1)
print(f'training with {len(dataSample)} instances')
#define X (data features) and y (target)
#X is all useful features
#y is target (final column 'default payment next month')
y = dataSample['damage_grade']
#Default Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade
features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

#Other features to drop:
dropColumns = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id'] #Too specific categorical, would produce thousands of columns
#I decided we keep geo_level_1_id as there are only 30 and it is overal region, may have more value
#but we need to make it categorical to generate one-hot encodings

# features['geo_level_1_id'] = features['geo_level_1_id'].astype('category')


#Region
# features = features.drop(dropColumns, axis = 1)
#Data adjustment
# encoder = OneHotEncoder(sparse_output=False, drop='first')

#These tables are in string type and must be converted
# categoryTables = ['geo_level_1_id','land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
categoryTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
one_hot_tables = pd.get_dummies(features[categoryTables], dtype=int, drop_first=True)




# scaler = MinMaxScaler()
# X = scaler.fit_transform(features)

#Now we replace our string tables with the encoded tables
X = features.drop(categoryTables, axis = 1)
# X = X.join(one_hot_tables)
X = pd.concat([X, one_hot_tables], axis = 1)




# encoder = TargetEncoder(target_type='auto')
# encoder.set_output(transform="pandas")

# # 3. Fit and transform
# X = encoder.fit_transform(X, y)


training with 244410 instances


In [11]:
#preparing eathquake data for PCA analysis
# scaler = StandardScaler()
# scaler = MinMaxScaler()
# X_scaled = pd.DataFrame(scaler.fit_transform(X))


# # pca = PCA(n_components=0.5)
# # X_pca = pca.fit_transform(X_scaled)

# # X = X_pca

# # mca = prince.MCA(n_components=0.7)
# mca = prince.MCA(
#         n_components=30,
#         n_iter=3,
#         copy=False,
#         check_input=False,
#         engine='sklearn',
#         random_state=42,
#         one_hot=False
#     )
# X_mca = mca.fit_transform(X_scaled)
# X = X_mca

In [12]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 244410 entries, 258521 to 207170
Data columns (total 60 columns):
 #   Column                                  Non-Null Count   Dtype
---  ------                                  --------------   -----
 0   geo_level_1_id                          244410 non-null  int64
 1   geo_level_2_id                          244410 non-null  int64
 2   geo_level_3_id                          244410 non-null  int64
 3   count_floors_pre_eq                     244410 non-null  int64
 4   age                                     244410 non-null  int64
 5   area_percentage                         244410 non-null  int64
 6   height_percentage                       244410 non-null  int64
 7   has_superstructure_adobe_mud            244410 non-null  int64
 8   has_superstructure_mud_mortar_stone     244410 non-null  int64
 9   has_superstructure_stone_flag           244410 non-null  int64
 10  has_superstructure_cement_mortar_stone  244410 non-null  int64
 11  

In [13]:
#grid search pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    # ('cat', TargetEncoder(), ['categorical_col1', 'categorical_col2']),
    # ('scaler', MinMaxScaler().set_output(transform="pandas")),#This keeps it as pandas dataframe instead of numpy array, needed for mca
    ('scaler', RobustScaler().set_output(transform="pandas")),
    # ('mca', prince.MCA(
    #     n_components=22,
    #     n_iter=3,
    #     copy=False,
    #     check_input=False,
    #     engine='sklearn',
    #     random_state=42,
    #     one_hot=False
    # )),
    ('classifier', RandomForestClassifier())
])

param_grid = {
    # 'classifier__n_estimators': [400],  # Correct prefix
    'classifier__n_estimators': [400, 500, 600],  # Correct prefix
    'classifier__max_depth': [None], # Correct prefix
    # ... other parameters
}

#This code was taken from the documentation
gridSearch = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='roc_auc_ovr', # Metric to optimize (e.g., accuracy, f1_macro, etc.)
    n_jobs=-1, # Use all available processors
    verbose = 100
)

In [14]:
gridSearch

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'classifier__max_depth': [None], 'classifier__n_estimators': [400, 500, ...]}"
,scoring,'roc_auc_ovr'
,n_jobs,-1
,refit,True
,cv,5
,verbose,100
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,with_centering,True


In [15]:
#Grid search training
start_time = time.perf_counter()

gridSearch.fit(X_train, y_train)

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.4f} seconds")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Elapsed time: 306.6363 seconds


In [16]:
print(f"Best parameters: {gridSearch.best_params_}")
print(f"Best cross-validation score: {gridSearch.best_score_:.4f}")

# Get the best model
best_model = gridSearch.best_estimator_

Best parameters: {'classifier__max_depth': None, 'classifier__n_estimators': 500}
Best cross-validation score: 0.8429


In [17]:
train_accuracy = best_model.score(X_train, y_train)
print(f"Train set accuracy with best model: {train_accuracy:.4f}")

Train set accuracy with best model: 1.0000


In [18]:
test_accuracy = best_model.score(X_test, y_test)
print(f"Test set accuracy with best model: {test_accuracy:.4f}")

Test set accuracy with best model: 0.7117


In [22]:
param_grid = {
    'classifier__n_estimators': [500, 600, 700],  # Correct prefix
    'classifier__max_depth': [None],
    # ... other parameters
}

#This code was taken from the documentation
gridSearch2 = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='accuracy', # Metric to optimize (e.g., accuracy, f1_macro, etc.)
    n_jobs=-1, # Use all available processors
    verbose = 1
)

In [23]:
#Grid search training
start_time = time.perf_counter()

gridSearch2.fit(X_train, y_train)

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.4f} seconds")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Elapsed time: 404.3930 seconds


In [24]:
print(f"Best parameters: {gridSearch2.best_params_}")
print(f"Best cross-validation score: {gridSearch2.best_score_:.4f}")

# Get the best model
best_model = gridSearch2.best_estimator_

Best parameters: {'classifier__max_depth': None, 'classifier__n_estimators': 700}
Best cross-validation score: 0.7121


In [25]:
train_accuracy = best_model.score(X_train, y_train)
print(f"Train set accuracy with best model: {train_accuracy:.4f}")

Train set accuracy with best model: 1.0000


In [26]:
test_accuracy = best_model.score(X_test, y_test)
print(f"Test set accuracy with best model: {test_accuracy:.4f}")

Test set accuracy with best model: 0.7147


## Gradient boosted trees

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [32]:
gb_pipeline = Pipeline([
    # ('cat', TargetEncoder(), ['categorical_col1', 'categorical_col2']),
    # ('scaler', MinMaxScaler().set_output(transform="pandas")),#This keeps it as pandas dataframe instead of numpy array, needed for mca
    ('scaler', RobustScaler().set_output(transform="pandas")),
    ('classifier', HistGradientBoostingClassifier(early_stopping=True, random_state=42))
])

gb_param_grid = {
    # 'learning_rate': [0.01, 0.1, 0.2],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__max_iter': [500, 600, 700, 800],
    'classifier__max_depth': [None],
    # 'max_leaf_nodes': [15, 31, 63], #Maximum leaves per tree. This is often more effective than max_depth for tree-based models.
    # 'min_samples_leaf': [20, 50], Minimum samples required in a leaf. Increasing this helps with overfitting.
    # 'l2_regularization': [0.0, 0.1, 1.0], #Regularization strength.
    # 'max_bins': [128, 255] #Maximum number of bins for histograms. Using fewer bins can act as a regularizer
    'classifier__max_bins': [255]
}

gb_gridSearch = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=5, # 5-fold cross-validation
    scoring='roc_auc_ovr', # Metric to optimize (e.g., accuracy, f1_macro, etc.)
    n_jobs=-1, # Use all available processors
    verbose = 1
)

In [33]:
#Grid search training
start_time = time.perf_counter()

gb_gridSearch.fit(X_train, y_train)

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.4f} seconds")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Elapsed time: 305.3624 seconds


In [34]:
print(f"Best parameters: {gb_gridSearch.best_params_}")
print(f"Best cross-validation score: {gb_gridSearch.best_score_:.4f}")

# Get the best model
best_model = gb_gridSearch.best_estimator_

Best parameters: {'classifier__learning_rate': 0.1, 'classifier__max_bins': 255, 'classifier__max_depth': None, 'classifier__max_iter': 600}
Best cross-validation score: 0.8625


In [35]:
train_accuracy = best_model.score(X_train, y_train)
print(f"Train set accuracy with best model: {train_accuracy:.4f}")

Train set accuracy with best model: 0.7872


In [36]:
test_accuracy = best_model.score(X_test, y_test)
print(f"Test set accuracy with best model: {test_accuracy:.4f}")

Test set accuracy with best model: 0.7332


### Second iteration

In [49]:
gb_pipeline = Pipeline([
    # ('cat', TargetEncoder(), ['categorical_col1', 'categorical_col2']),
    # ('scaler', MinMaxScaler().set_output(transform="pandas")),#This keeps it as pandas dataframe instead of numpy array, needed for mca
    ('scaler', RobustScaler().set_output(transform="pandas")),
    ('classifier', HistGradientBoostingClassifier(early_stopping=True, random_state=42))
])

gb_param_grid = {
    # 'learning_rate': [0.01, 0.1, 0.2],
    'classifier__learning_rate': [0.05, 0.1, 0.15],
    'classifier__max_iter': [600, 700, 800],
    'classifier__max_depth': [None],
    # 'max_leaf_nodes': [15, 31, 63], #Maximum leaves per tree. This is often more effective than max_depth for tree-based models.
    # 'min_samples_leaf': [20, 50], Minimum samples required in a leaf. Increasing this helps with overfitting.
    'classifier__l2_regularization': [0.0, 0.01, 0.1], #Regularization strength.
    # 'max_bins': [128, 255] #Maximum number of bins for histograms. Using fewer bins can act as a regularizer
    'classifier__max_bins': [255]
}

gb_gridSearch2 = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=5, # 5-fold cross-validation
    scoring='f1_macro', # Metric to optimize (e.g., accuracy, f1_macro, etc.)
    n_jobs=-1, # Use all available processors
    verbose = 100
)

In [ ]:
#Grid search training
start_time = time.perf_counter()

gb_gridSearch2.fit(X_train, y_train)

end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.4f} seconds")

Fitting 5 folds for each of 27 candidates, totalling 135 fits


In [ ]:
print(f"Best parameters: {gb_gridSearch2.best_params_}")
print(f"Best cross-validation score: {gb_gridSearch2.best_score_:.4f}")

# Get the best model
best_model = gb_gridSearch2.best_estimator_

In [ ]:
train_accuracy = best_model.score(X_train, y_train)
print(f"Train set accuracy with best model: {train_accuracy:.4f}")

In [ ]:
test_accuracy = best_model.score(X_test, y_test)
print(f"Test set accuracy with best model: {test_accuracy:.4f}")